# RSHazeNet - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains RSHazeNet on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: RSHazeNet  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [1]:
#@title 1. Install Dependencies
!pip install timm pytorch-msssim lpips scikit-image thop gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.8 MB/s eta 0:00:00


In [2]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/weights"

Mounted at /content/drive


In [3]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

# Remove corrupted files if they exist
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

--2026-06-22 00:25:04--  https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.65.18, 2620:100:6021:18::a27d:4112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.65.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1 [following]
--2026-06-22 00:25:05--  https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucbfdebfb83982ad5684b6ac6443.dl.dropboxusercontent.com/cd/0/inline/DC2Y9G5_2IxrTv-uhG4X1Rsup_62WDaU1lgd78Vzc_6Gz3nFWpGVXkrn1L7bxTMmMVcnjUB39x6wQiHpYt9zIDXnfYgfIJCC7DuyhKqp2OVo2CxXwdA-cqF74yIJaaQO1suKmresVrbYIxSlGidapJUL/file?dl=1# [following]
--2026-06-22 00:25:06--  https://ucbfdebfb83982ad5684b6ac6443.dl.dropboxusercontent.

In [4]:
#@title 4. Clone RSHazeNet Repository
!git clone https://github.com/sumire25/RSHazeNet.git
%cd RSHazeNet
!ls

Cloning into 'RSHazeNet'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 28 (delta 6), reused 28 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 2.81 MiB | 7.61 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/RSHazeNet
datasets.py  images    model.py    pytorch_msssim  RSHazeNet.ipynb  train.py
evaluate.py  infer.py  options.py  README.md	   test.py	    utils.py


In [5]:
#@title 5. Download Pretrained Weights
!gdown "1Leyg1sw4x48wEo5zsPKBGtVaCF5NWdY_" -O rs_haze.pth
!ls -lh rs_haze.pth

Downloading...
From: https://drive.google.com/uc?id=1Leyg1sw4x48wEo5zsPKBGtVaCF5NWdY_
To: /content/RSHazeNet/rs_haze.pth
100% 4.82M/4.82M [00:00<00:00, 27.1MB/s]
-rw-r--r-- 1 root root 4.6M Jul 21  2023 rs_haze.pth


In [6]:
#@title 6. Compute FLOPs and Parameters
import torch
from thop import profile
from model import RSHazeNet

model = RSHazeNet().cuda()
model.eval()

dummy_input = torch.randn(1, 3, 512, 512).cuda()
macs, params = profile(model, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- RSHazeNet Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pixelshuffle.PixelShuffle'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.

--- RSHazeNet Complexity ---
FLOPs: 40.1427 G
Parameters: 1.1900 M


## Phase 1: Test Pretrained Weights

In [7]:
#@title 7. Run Inference with Pretrained Weights
!python infer.py \
  --test_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  --test_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/ \
  --weights rs_haze.pth \
  --result_path /content/drive/MyDrive/RSHazeNet_Results/pretrained_results/

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Inference: 100% 45/45 [00:34<00:00,  1.32it/s]


In [8]:
#@title 8. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/RSHazeNet_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 128MB/s]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 3.188
ERGAS: 18.521
UIQI: 0.838
QNR: 

## Phase 2: Train from Scratch

In [10]:
#@title 9. Train RSHazeNet
!python train.py \
  --train_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/input/ \
  --train_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/target/ \
  --val_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/input/ \
  --val_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/target/ \
  --epochs 100 \
  --lr 0.0002 \
  --batch_size_train 8 \
  --batch_size_val 8 \
  --patch_size_train 256 \
  --patch_size_val 256 \
  --val_freq 3 \
  --save_freq 20 \
  --save_dir /content/drive/MyDrive/RSHazeNet_Results/weights/

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
-------------------------------------------------------------------------------------------------------
/content/RSHazeNet/train.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', ar

## Phase 3: Test Trained Weights

In [11]:
#@title 10. Run Inference with Trained Weights
!python infer.py \
  --test_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  --test_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/ \
  --weights /content/drive/MyDrive/RSHazeNet_Results/weights/model_best.pth \
  --result_path /content/drive/MyDrive/RSHazeNet_Results/trained_results/

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Inference: 100% 45/45 [00:06<00:00,  7.24it/s]


In [12]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/RSHazeNet_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 3.321
ERGAS: 25.309
UIQI: 0.781
QNR: 0.558
BRISQUE: 33.967
NIQE: 1.781
Histogram: 0.796
Spectral Curve Deviation: 0.888
PSNR: 17.552
SSIM: 0.781
CIEDE2000: 10.519
LPIPS: 0.254
FID: 91.932
